# PyGraphDB Property Graph API

This notebook demonstrates the current PyGraphDB API for building and querying a small property graph. It uses the in-memory backend so it runs without optional native dependencies.

## Local Development Import

When running this notebook from the repository checkout, add `../src` to `sys.path`. If you installed the package with `uv sync` or from a wheel, this cell is harmless.

In [ ]:
from pathlib import Path
import sys

repo_src = Path.cwd().parent / 'src'
if repo_src.exists():
    sys.path.insert(0, str(repo_src))

In [ ]:
from graphdb import ConstraintError, Edge, GraphDB, Node
from kvstores import InMemoryKVStore
from serializers import MessagePackSerializer

## Create a Graph

A property graph has nodes with labels/properties and directed edges with types/properties.

In [ ]:
graph = GraphDB(InMemoryKVStore(), MessagePackSerializer())

graph.put_node(Node('alice', labels=['Person', 'Employee'], properties={'name': 'Alice', 'age': 30}))
graph.put_node(Node('bob', labels=['Person'], properties={'name': 'Bob', 'age': 34}))
graph.put_node(Node('carol', labels=['Person'], properties={'name': 'Carol', 'age': 29}))
graph.put_node(Node('acme', labels=['Company'], properties={'name': 'Acme Corp'}))

graph.put_edge(Edge('alice-knows-bob', source='alice', target='bob', type='KNOWS', properties={'since': 2020}))
graph.put_edge(Edge('bob-knows-carol', source='bob', target='carol', type='KNOWS', properties={'since': 2021}))
graph.put_edge(Edge('alice-works-at-acme', source='alice', target='acme', type='WORKS_AT', properties={'role': 'Engineer'}))

graph.count_nodes(), graph.count_edges()

## Fetch Nodes and Edges by ID

In [ ]:
alice = graph.get_node('alice')
edge = graph.get_edge('alice-knows-bob')

alice.to_dict(), edge.to_dict()

## Property Graph Lookups

The in-memory and LMDB backends maintain exact-match indexes for labels, edge types, and hashable property values.

In [ ]:
people = graph.nodes_by_label('Person')
employees = graph.find_nodes(labels=['Employee'])
age_34 = graph.find_nodes(properties={'age': 34})
knows_edges = graph.edges_by_type('KNOWS')
alice_edges = graph.find_edges(source='alice')

{
    'people': [node.get_id for node in people],
    'employees': [node.get_id for node in employees],
    'age_34': [node.get_id for node in age_34],
    'knows_edges': [edge.get_id for edge in knows_edges],
    'alice_edges': [edge.get_id for edge in alice_edges],
}

## Traversal

Use adjacency APIs for neighbors, directional degrees, incident edges, and bounded BFS.

In [ ]:
{
    'alice_out_neighbors': graph.neighbors('alice', direction='out'),
    'bob_in_neighbors': graph.neighbors('bob', direction='in'),
    'alice_out_degree': graph.out_degree('alice'),
    'bob_in_degree': graph.in_degree('bob'),
    'bfs_from_alice': graph.bfs('alice', direction='out', max_depth=2),
}

## Updating Labels, Properties, and Edge Types

In [ ]:
graph.add_label('bob', 'Employee')
graph.set_node_property('bob', 'department', 'Sales')
graph.set_edge_property('alice-knows-bob', 'strength', 'high')
graph.rename_edge_type('WORKS_AT', 'EMPLOYED_BY')

{
    'bob_labels': sorted(graph.get_node('bob').labels),
    'bob_properties': graph.node_properties('bob'),
    'strong_edges': [edge.get_id for edge in graph.find_edges(properties={'strength': 'high'})],
    'employed_by': [edge.get_id for edge in graph.edges_by_type('EMPLOYED_BY')],
}

## Deletion Semantics

By default node deletion is restrictive: a node with incident edges cannot be deleted. Use `mode='detach'` to delete incident edges first.

In [ ]:
try:
    graph.delete_node('bob')
except ConstraintError as exc:
    print(exc)

graph.delete_node('carol', mode='detach')
graph.has_node('carol'), graph.has_edge('bob-knows-carol')

## Integrity and Maintenance

`check_integrity()` validates that edges reference existing endpoints and that adjacency entries match edge records. `rebuild_indexes()` reconstructs backend-maintained indexes and optimized adjacency entries from node/edge records.

In [ ]:
integrity = graph.check_integrity()
print(integrity)

graph.rebuild_indexes()
graph.check_integrity()

## Optional LMDB Persistence

Install the optional LMDB backend with `uv sync --extra lmdb`. The cell below is safe to run when LMDB is not installed; it will print a message and skip.

In [ ]:
import shutil
import tempfile

try:
    from kvstores import LMDBStore

    path = tempfile.mkdtemp(prefix='pygraphdb_notebook_')
    persistent_graph = GraphDB(LMDBStore(path=path), MessagePackSerializer())
    persistent_graph.put_node(Node('a', labels=['Example'], properties={'name': 'A'}))
    persistent_graph.put_node(Node('b', labels=['Example'], properties={'name': 'B'}))
    persistent_graph.put_edge(Edge('ab', source='a', target='b', type='LINKS'))
    persistent_graph.close()

    reopened = GraphDB(LMDBStore(path=path), MessagePackSerializer())
    print(reopened.neighbors('a'))
    print([node.get_id for node in reopened.find_nodes(labels=['Example'])])
    reopened.close()
    shutil.rmtree(path, ignore_errors=True)
except ModuleNotFoundError as exc:
    print(f'Skipping LMDB example: {exc}')